<h1><center>Credit Card Fraud Detection Project</center></h1>
<hr>

<h1>Objective</h1>
<p>
The goal of this project is to build a machine learning model that can accurately identify 
fraudulent credit card transactions from a highly imbalanced dataset. The dataset contains 
transactions made by European cardholders over two days in September 2013, with 284,807 
transactions in total, of which only 492 are fraudulent (~0.172% of all transactions).
</p>
<p>
Due to confidentiality, most input features (V1–V28) have been transformed using PCA and are 
not directly interpretable, while <code>Time</code> and <code>Amount</code> remain in their 
original form. The target variable, <code>Class</code>, is 1 for fraud and 0 for a normal 
transaction.
</p>
<p>
Because of the severe class imbalance, standard accuracy is not a meaningful metric here. 
This project focuses on:
</p>
<ul>
  <li>Exploring and understanding the class imbalance</li>
  <li>Handling imbalanced data (e.g. undersampling, oversampling with SMOTE, class weights)</li>
  <li>Training and comparing classification models (Logistic Regression, Random Forest, XGBoost, etc.)</li>
  <li>Evaluating performance using Precision, Recall, F1-score, and AUPRC/ROC-AUC rather than accuracy alone</li>
  <li>Minimizing false negatives (missed fraud) while keeping false positives manageable</li>
</ul>
<hr>

**Importing the dependencies**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

**Data Preprocessing and Analysis**

In [2]:
#loading the dataset
credit_card_df = pd.read_csv('creditcard.csv')

In [3]:
credit_card_df.sample(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
50908,44703.0,1.230690,0.266259,0.414562,0.783359,-0.647366,-1.290339,0.006785,-0.151158,0.206718,...,-0.280581,-0.876754,0.184438,0.635982,0.128344,0.078468,-0.031303,0.031492,2.67,0
209652,137651.0,0.509274,0.391497,-1.332871,-1.960396,1.106500,-1.216765,1.338558,-0.576177,-1.834096,...,0.532142,1.550145,-0.451706,0.589325,0.643138,0.207221,-0.057730,-0.035346,20.00,0
133291,80322.0,1.372116,-0.734432,0.891268,-0.558316,-1.179410,0.129974,-1.020251,0.012166,-0.170935,...,-0.516360,-0.772485,0.019108,-0.391211,0.159241,1.007300,0.000606,0.017121,13.90,0
202643,134427.0,1.218462,-2.778199,-0.175982,-0.280002,-2.040337,0.596023,-1.160939,0.187592,0.792407,...,0.426102,0.557074,-0.215369,-0.266426,-0.466460,-0.225986,-0.020148,0.027228,448.63,0
168945,119450.0,0.109713,1.280614,-2.746476,0.426289,1.351428,-1.657189,0.742286,0.126770,0.000223,...,-0.012638,0.004441,-0.094128,-0.999588,-0.009680,-0.453708,0.165608,-0.066239,30.00,0


In [5]:
credit_card_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [6]:
#distribution of legit vs fraudulent transactions
credit_card_df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

This dataset is highly imbalanced.

0 --> Normal Transaction

1 --> Fraudulent Transaction

In [10]:
#seperating the data for analysis
legit = credit_card_df[credit_card_df.Class == 0]
fraudulent = credit_card_df[credit_card_df.Class == 1]

In [11]:
print(legit.shape)
print(fraudulent.shape)

(284315, 31)
(492, 31)


In [12]:
#statistical measures of the data
legit.Amount.describe()

count    284315.000000
mean         88.291022
std         250.105092
min           0.000000
25%           5.650000
50%          22.000000
75%          77.050000
max       25691.160000
Name: Amount, dtype: float64

In [13]:
fraudulent.Amount.describe()

count     492.000000
mean      122.211321
std       256.683288
min         0.000000
25%         1.000000
50%         9.250000
75%       105.890000
max      2125.870000
Name: Amount, dtype: float64

In [14]:
#compare the values for both transactions
credit_card_df.groupby('Class').mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,94838.202258,0.008258,-0.006271,0.012171,-0.007860,0.005453,0.002419,0.009637,-0.000987,0.004467,...,-0.000644,-0.001235,-0.000024,0.000070,0.000182,-0.000072,-0.000089,-0.000295,-0.000131,88.291022
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


**Under-sampling**

Build a sample dataset containing similar distribution of normal transactions and fraudulent transactions.

Number of fraudulent transactions -> 492

In [15]:
legit_sample = legit.sample(492)

Concatenating 2 dataframes

In [17]:
new_dataset = pd.concat([legit_sample, fraudulent], axis=0)

In [21]:
new_dataset.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
139687,83293.0,-0.406183,1.065048,1.567251,-0.036176,-0.062886,-0.890446,0.713926,-0.072926,-0.412375,...,-0.228129,-0.585730,-0.027613,0.363989,-0.190374,0.071156,0.266641,0.120983,1.78,0
79634,58103.0,-0.239472,0.346865,2.287737,0.391006,-0.200371,-0.503093,0.516090,-0.365123,0.941756,...,0.060721,0.652898,0.087180,0.693116,-1.196098,-0.844599,-0.179323,-0.168799,9.99,0
248701,154055.0,1.978772,-0.358727,-0.273964,0.492375,-0.681763,-0.515495,-0.537176,-0.105861,1.347888,...,0.196474,0.860857,0.087436,-0.061092,-0.049398,-0.209352,0.046563,-0.035256,13.53,0
129585,79145.0,-1.362459,-1.387320,1.769497,0.888551,-2.611663,1.384822,1.932643,0.065893,0.737467,...,0.230436,0.057716,1.304614,0.408513,0.215923,0.430532,-0.148666,0.076107,629.90,0
234358,147941.0,1.395936,-1.338106,-1.638983,0.066545,0.324173,1.140760,-0.003052,0.295086,0.926949,...,0.081206,-0.070575,-0.020984,-1.619183,-0.343631,-0.003374,-0.033955,-0.037534,273.99,0


In [22]:
new_dataset.tail()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
279863,169142.0,-1.927883,1.125653,-4.518331,1.749293,-1.566487,-2.010494,-0.882850,0.697211,-2.064945,...,0.778584,-0.319189,0.639419,-0.294885,0.537503,0.788395,0.292680,0.147968,390.00,1
280143,169347.0,1.378559,1.289381,-5.004247,1.411850,0.442581,-1.326536,-1.413170,0.248525,-1.127396,...,0.370612,0.028234,-0.145640,-0.081049,0.521875,0.739467,0.389152,0.186637,0.76,1
280149,169351.0,-0.676143,1.126366,-2.213700,0.468308,-1.120541,-0.003346,-2.234739,1.210158,-0.652250,...,0.751826,0.834108,0.190944,0.032070,-0.739695,0.471111,0.385107,0.194361,77.89,1
281144,169966.0,-3.113832,0.585864,-5.399730,1.817092,-0.840618,-2.943548,-2.208002,1.058733,-1.632333,...,0.583276,-0.269209,-0.456108,-0.183659,-0.328168,0.606116,0.884876,-0.253700,245.00,1
281674,170348.0,1.991976,0.158476,-2.583441,0.408670,1.151147,-0.096695,0.223050,-0.068384,0.577829,...,-0.164350,-0.295135,-0.072173,-0.450261,0.313267,-0.289617,0.002988,-0.015309,42.53,1


In [23]:
new_dataset['Class'].value_counts()

Class
0    492
1    492
Name: count, dtype: int64

In [24]:
new_dataset.groupby('Class').mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,93497.022358,-0.078886,-0.098287,0.006264,-0.044285,0.000466,-0.004490,-0.017160,-0.050059,0.015811,...,-0.011127,0.010078,0.020260,0.015431,0.004341,0.002786,-0.021510,-0.022746,0.022328,96.791951
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


**Splitting data into features and targets**

In [25]:
X = new_dataset.drop(columns='Class')
y = new_dataset['Class']

In [26]:
X

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
139687,83293.0,-0.406183,1.065048,1.567251,-0.036176,-0.062886,-0.890446,0.713926,-0.072926,-0.412375,...,0.100283,-0.228129,-0.585730,-0.027613,0.363989,-0.190374,0.071156,0.266641,0.120983,1.78
79634,58103.0,-0.239472,0.346865,2.287737,0.391006,-0.200371,-0.503093,0.516090,-0.365123,0.941756,...,-0.108707,0.060721,0.652898,0.087180,0.693116,-1.196098,-0.844599,-0.179323,-0.168799,9.99
248701,154055.0,1.978772,-0.358727,-0.273964,0.492375,-0.681763,-0.515495,-0.537176,-0.105861,1.347888,...,-0.154239,0.196474,0.860857,0.087436,-0.061092,-0.049398,-0.209352,0.046563,-0.035256,13.53
129585,79145.0,-1.362459,-1.387320,1.769497,0.888551,-2.611663,1.384822,1.932643,0.065893,0.737467,...,1.340973,0.230436,0.057716,1.304614,0.408513,0.215923,0.430532,-0.148666,0.076107,629.90
234358,147941.0,1.395936,-1.338106,-1.638983,0.066545,0.324173,1.140760,-0.003052,0.295086,0.926949,...,0.203107,0.081206,-0.070575,-0.020984,-1.619183,-0.343631,-0.003374,-0.033955,-0.037534,273.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279863,169142.0,-1.927883,1.125653,-4.518331,1.749293,-1.566487,-2.010494,-0.882850,0.697211,-2.064945,...,1.252967,0.778584,-0.319189,0.639419,-0.294885,0.537503,0.788395,0.292680,0.147968,390.00
280143,169347.0,1.378559,1.289381,-5.004247,1.411850,0.442581,-1.326536,-1.413170,0.248525,-1.127396,...,0.226138,0.370612,0.028234,-0.145640,-0.081049,0.521875,0.739467,0.389152,0.186637,0.76
280149,169351.0,-0.676143,1.126366,-2.213700,0.468308,-1.120541,-0.003346,-2.234739,1.210158,-0.652250,...,0.247968,0.751826,0.834108,0.190944,0.032070,-0.739695,0.471111,0.385107,0.194361,77.89
281144,169966.0,-3.113832,0.585864,-5.399730,1.817092,-0.840618,-2.943548,-2.208002,1.058733,-1.632333,...,0.306271,0.583276,-0.269209,-0.456108,-0.183659,-0.328168,0.606116,0.884876,-0.253700,245.00


In [27]:
y

139687    0
79634     0
248701    0
129585    0
234358    0
         ..
279863    1
280143    1
280149    1
281144    1
281674    1
Name: Class, Length: 984, dtype: int64

**train_test_split**

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [29]:
print(X.shape, X_train.shape, X_test.shape)

(984, 30) (787, 30) (197, 30)


**Model Training and Evaluation**

In [30]:
model = LogisticRegression()

In [ ]:
#training model with training data
model.fit(X_train, y_train)

In [33]:
#evaluation on accuracy of training data
X_train_pred = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_pred, y_train)
print('Training data accuracy is:' ,training_data_accuracy)

Training data accuracy is: 0.940279542566709


In [34]:
#evaluation on accuracy of testing data
X_test_pred = model.predict(X_test)
testing_data_accuracy = accuracy_score(X_test_pred, y_test)
print('Testing data accuracy score:' ,testing_data_accuracy)

Testing data accuracy score: 0.934010152284264
